In [1]:
from tqdm import tqdm
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd
from bertopic import BERTopic
from konlpy.tag import Okt
import numpy as np
from umap import UMAP
import re
import datetime
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora.dictionary import Dictionary
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
from scipy.spatial.distance import squareform
try:
    from collections.abc import Mapping
except ImportError:
    from collections import Mapping
    
from gensim.models import Word2Vec
from scipy.cluster import hierarchy as sch
import warnings
warnings.filterwarnings('ignore')
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
stop_word_file = "../Datasets/RawDatasets/txt/new_stopwords.txt"
stop_word_file2 = "../Datasets/RawDatasets/txt/stopwords.txt"
stop_words1 =  [line.strip() for line in open(stop_word_file, encoding="utf-8").readlines()]
stop_words2 =  [line.strip() for line in open(stop_word_file2, encoding="utf-8").readlines()]
stop_words = stop_words1+stop_words2

In [3]:
class CustomTokenizer:
    def __init__(self, tagger):
        self.tagger = tagger
    def __call__(self, sent):
        #sent = sent[:1000000]
        hangul = re.compile('[^ 0-9가-힣+]')
        sent = hangul.sub(' ', sent)
        sent = " ".join(sent.split())
        word_tokens = self.tagger.pos(sent, stem=True)
        temp = [word[0] for word in word_tokens if (word[1] =='Adjective' or  word[1] =='Noun')]
        result = [word for word in temp if (len(word) > 1  and ( not word in stop_words))]
        #한 단어짜리 토큰 제거
        #불용어 제거
        return result

In [4]:
def get_word_embedding_based_similarity(topic1, topic2):
    similarity = 0
    for i in topic1:
        for j in topic2:
            similarity+= word2vec_model.wv.similarity(i,j)
    return similarity/(len(topic1)*len(topic2))

In [5]:
def get_coherence(mdl,k):
    topics = []
    for i in range(k):
        topics.append([word[0] for word in mdl.get_topic(i)])

    cm = CoherenceModel(topics = topics,
                   #corpus = corpus,
                    texts =texts,
                    dictionary = dictionary,
                   coherence = 'c_npmi')
    return cm.get_coherence()

In [6]:
data = pd.read_feather('../Datasets/dataset.ftr')
texts =data.okt
dictionary = Dictionary(texts)
corpus = [dictionary.doc2bow(text) for text in texts]

In [7]:
word2vec_model = Word2Vec.load('../models/word2vec.model')

In [8]:
from sentence_transformers import SentenceTransformer
sentence_model = SentenceTransformer("jhgan/ko-sroberta-multitask")
umap_model = UMAP(n_neighbors=15, n_components=5, 
                  min_dist=0.0, metric='cosine', random_state=33)
custom_tokenizer = CustomTokenizer(Okt())
vectorizer = CountVectorizer(tokenizer = custom_tokenizer)


In [10]:
score_list=[]
for sample_num in tqdm(range(5)):
    docs = pd.read_feather(f'../Datasets/sampledf_{sample_num}.ftr').corrected_twit.to_list()
    model = BERTopic(language = 'Korean',
             top_n_words=20,
             nr_topics= "auto",
             embedding_model = sentence_model,
             umap_model = umap_model,
             vectorizer_model = vectorizer,
             ) 
    topic, prob = model.fit_transform(docs)
    hier_topics = model.hierarchical_topics(docs)
    
    hier_topics['words'] = range(len(hier_topics))
    hier_topics['lc_words'] = range(len(hier_topics))
    hier_topics['rc_words'] = range(len(hier_topics))
    for i in range(len(hier_topics)):
        hier_topics['lc_words'][i] = [word for word in hier_topics['Child_Left_Name'][i].split('_') if (word !='')]
        hier_topics['rc_words'][i] = [word for word in hier_topics['Child_Right_Name'][i].split('_') if (word !='')]
        hier_topics['words'][i] = [word for word in hier_topics['Parent_Name'][i].split('_') if (word !='')]

    similarity = 0
    diversity = 0
    for i in range(len(hier_topics)):
        #similarity
        similarity+=get_word_embedding_based_similarity(hier_topics.loc[i].words,hier_topics.loc[i].lc_words)
        similarity+=get_word_embedding_based_similarity(hier_topics.loc[i].words,hier_topics.loc[i].rc_words)
        #diversity

        diversity += (1 - get_word_embedding_based_similarity(hier_topics.loc[i].lc_words,hier_topics.loc[i].rc_words))

        #print("#unique words :", len(unique_words))
        #print("#lc+rc : ",len(lc)+len(rc))

    similarity /=(len(hier_topics)*2)
    diversity /= len(hier_topics)
    
    coherence = get_coherence(model, len(model.get_topic_info())-1)
    score = (0.3 *(similarity) + 0.3*(diversity)+0.4*(coherence))
    temp = [similarity,diversity,coherence,score]
    print(temp)
    score_list.append(temp)

 20%|█████████████▌                                                      | 1/5 [23:08<1:32:35, 1388.78s/it]

[0.7069489427588582, 0.3595521632535716, -0.18297085170714011, 0.24676199112087288]



 40%|███████████████████████████▏                                        | 2/5 [48:15<1:12:54, 1458.14s/it]

[0.7070657045431937, 0.3771649835001977, -0.19257501464274063, 0.2482392005559212]



 60%|████████████████████████████████████████▊                           | 3/5 [1:18:38<54:09, 1624.69s/it]

[0.737959635708759, 0.3179611506760468, -0.19482454303546426, 0.23884641870125603]



 80%|██████████████████████████████████████████████████████▍             | 4/5 [1:47:29<27:46, 1666.74s/it]

[0.6564579647261805, 0.4157527748676948, -0.17296221495951564, 0.2524783358943563]



100%|████████████████████████████████████████████████████████████████████| 5/5 [2:15:06<00:00, 1621.29s/it]

[0.6239708630915266, 0.44444762231432844, -0.14863741519039303, 0.2610705795455993]


In [19]:
scores = [s[3] for s in score_list]

In [22]:
print(((score_list[0][0])+(score_list[1][0])+(score_list[2][0])+(score_list[3][0])+(score_list[4][0]))/5)
print(((score_list[0][1])+(score_list[1][1])+(score_list[2][1])+(score_list[3][1])+(score_list[4][1]))/5)
print(((score_list[0][2])+(score_list[1][2])+(score_list[2][2])+(score_list[3][2])+(score_list[4][2]))/5)
print(((score_list[0][3])+(score_list[1][3])+(score_list[2][3])+(score_list[3][3])+(score_list[4][3]))/5)

0.6864806221657036
0.3829757389223679
-0.17839400790705073
0.24947930516360114


# Interpretation

In [31]:
docs = pd.read_feather(f'../Datasets/sampledf_{np.argmax(scores)}.ftr').corrected_twit.to_list()
model = BERTopic(language = 'Korean',
         top_n_words=20,
         nr_topics= "auto",
         embedding_model = sentence_model,
         umap_model = umap_model,
         vectorizer_model = vectorizer,
         ) 
topic, prob = model.fit_transform(docs)


In [32]:
hier_topics = model.hierarchical_topics(docs)

hier_topics['words'] = range(len(hier_topics))
hier_topics['lc_words'] = range(len(hier_topics))
hier_topics['rc_words'] = range(len(hier_topics))
for i in range(len(hier_topics)):
    hier_topics['lc_words'][i] = [word for word in hier_topics['Child_Left_Name'][i].split('_') if (word !='')]
    hier_topics['rc_words'][i] = [word for word in hier_topics['Child_Right_Name'][i].split('_') if (word !='')]
    hier_topics['words'][i] = [word for word in hier_topics['Parent_Name'][i].split('_') if (word !='')]

100%|███████████████████████████████████████████████████████████████████| 32/32 [00:01<00:00, 31.78it/s]


In [33]:
tree = model.get_topic_tree(hier_topics)

In [34]:
print(tree)

.
├─혈통_아프다_생리_부작용_예약
│    ├─혈통_아프다_생리_부작용_예약
│    │    ├─생리_부작용_생리통_정부_쓰레기
│    │    │    ├─부작용_식욕_피곤하다_엄청나다_불면증
│    │    │    │    ├─■──식욕_고프다_햄버거_부작용_폭발 ── Topic: 7
│    │    │    │    └─부작용_피곤하다_불면증_종일_수면제
│    │    │    │         ├─■──피곤하다_부작용_불면증_종일_수면제 ── Topic: 4
│    │    │    │         └─■──부작용_혓바늘_자건_자력_스킬 ── Topic: 31
│    │    │    └─생리_생리통_정부_쓰레기_국민
│    │    │         ├─■──생리_생리통_출혈_생리주기_생리전증후군 ── Topic: 3
│    │    │         └─■──정부_국민_쓰레기_미국_기자 ── Topic: 1
│    │    └─혈통_아프다_예약_완료_추가
│    │         ├─혈통_아프다_예약_완료_추가
│    │         │    ├─■──졸지_아프다_날씨_혈통_달성 ── Topic: 28
│    │         │    └─혈통_아프다_예약_완료_추가
│    │         │         ├─■──혈통_예약_완료_추가_아프다 ── Topic: 0
│    │         │         └─■──멀쩡하다_아프다_괜찮다_증상_다행 ── Topic: 2
│    │         └─■──차갑다_아프다_힘들다_후유증_가다실 ── Topic: 5
│    └─엄마_스타일_알레르기_언니_몸살
│         ├─■──엄마_알레르기_언니_가족_미화 ── Topic: 10
│         └─■──엄마_스타일_서도_독하다_두드러기 ── Topic: 13
└─겨드랑이_탈모_뽀로로_부루펜_비아그라
     ├─겨드랑이_탈모_뽀로로_부루펜_비아그라
     │    ├─부루펜_타이레놀_미리_괜찮다_금방

In [35]:
hier_topics

,Parent_ID,Parent_Name,Topics,Child_Left_ID,Child_Left_Name,Child_Right_ID,Child_Right_Name,Distance,words,lc_words,rc_words
31,64,혈통_아프다_부작용_생리_예약,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...",62,혈통_아프다_생리_부작용_예약,63,겨드랑이_탈모_뽀로로_부루펜_비아그라,1.310869,"[혈통, 아프다, 부작용, 생리, 예약]","[혈통, 아프다, 생리, 부작용, 예약]","[겨드랑이, 탈모, 뽀로로, 부루펜, 비아그라]"
30,63,겨드랑이_탈모_뽀로로_부루펜_비아그라,"[6, 8, 9, 11, 12, 14, 15, 16, 17, 18, 19, 20, ...",60,겨드랑이_탈모_뽀로로_부루펜_비아그라,35,무섭다_무서움_폭력_처형_무경,1.271215,"[겨드랑이, 탈모, 뽀로로, 부루펜, 비아그라]","[겨드랑이, 탈모, 뽀로로, 부루펜, 비아그라]","[무섭다, 무서움, 폭력, 처형, 무경]"
29,62,혈통_아프다_생리_부작용_예약,"[0, 1, 2, 3, 4, 5, 7, 10, 13, 28, 31]",61,혈통_아프다_생리_부작용_예약,33,엄마_스타일_알레르기_언니_몸살,1.228505,"[혈통, 아프다, 생리, 부작용, 예약]","[혈통, 아프다, 생리, 부작용, 예약]","[엄마, 스타일, 알레르기, 언니, 몸살]"
28,61,혈통_아프다_생리_부작용_예약,"[0, 1, 2, 3, 4, 5, 7, 28, 31]",49,생리_부작용_생리통_정부_쓰레기,38,혈통_아프다_예약_완료_추가,1.139165,"[혈통, 아프다, 생리, 부작용, 예약]","[생리, 부작용, 생리통, 정부, 쓰레기]","[혈통, 아프다, 예약, 완료, 추가]"
27,60,겨드랑이_탈모_뽀로로_부루펜_비아그라,"[6, 8, 9, 11, 12, 14, 15, 16, 17, 18, 19, 20, ...",39,부루펜_타이레놀_미리_괜찮다_금방,59,겨드랑이_탈모_뽀로로_비아그라_이재명,1.109596,"[겨드랑이, 탈모, 뽀로로, 부루펜, 비아그라]","[부루펜, 타이레놀, 미리, 괜찮다, 금방]","[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]"
26,59,겨드랑이_탈모_뽀로로_비아그라_이재명,"[6, 8, 9, 12, 14, 15, 16, 17, 18, 19, 20, 22, ...",58,겨드랑이_탈모_뽀로로_비아그라_이재명,40,추가_남편_가족_아빠_킨님,1.073156,"[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[추가, 남편, 가족, 아빠, 킨님]"
25,58,겨드랑이_탈모_뽀로로_비아그라_이재명,"[6, 8, 9, 12, 14, 15, 16, 19, 20, 22, 24, 25, ...",48,고맙다_무덥다_외치_지화_축하,57,겨드랑이_탈모_뽀로로_비아그라_이재명,1.069516,"[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[고맙다, 무덥다, 외치, 지화, 축하]","[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]"
24,57,겨드랑이_탈모_뽀로로_비아그라_이재명,"[8, 12, 14, 16, 19, 20, 22, 24, 25, 27, 29, 30...",56,겨드랑이_탈모_뽀로로_비아그라_이재명,45,커피_진정하다_카페인_심장_위반,1.045999,"[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[커피, 진정하다, 카페인, 심장, 위반]"
23,56,겨드랑이_탈모_뽀로로_비아그라_이재명,"[8, 12, 14, 16, 19, 20, 22, 25, 27, 30, 32]",44,비아그라_회사_묵음_스펠링_이름,55,겨드랑이_탈모_뽀로로_이재명_밴드,1.039266,"[겨드랑이, 탈모, 뽀로로, 비아그라, 이재명]","[비아그라, 회사, 묵음, 스펠링, 이름]","[겨드랑이, 탈모, 뽀로로, 이재명, 밴드]"
22,55,겨드랑이_탈모_뽀로로_이재명_밴드,"[8, 12, 14, 16, 19, 20, 25, 27, 32]",54,겨드랑이_뽀로로_밴드_조종_삼겹살,46,탈모_이재명_상담_부작용_죄명,1.029846,"[겨드랑이, 탈모, 뽀로로, 이재명, 밴드]","[겨드랑이, 뽀로로, 밴드, 조종, 삼겹살]","[탈모, 이재명, 상담, 부작용, 죄명]"
